# Eval Baseline TrOCR (microsoft/trocr-large-handwritten)
Dataset: Teklia/IAM-line — test[:1000]
Metrik: CER & WER (apple-to-apple dengan DeepSeek OCR2)

## Installation

In [ ]:
%%capture
!pip install transformers==4.56.2 torch torchvision pillow
!pip install "datasets==4.3.0"
!pip install jiwer
!pip install sentencepiece


## Import

In [ ]:
import re
import os
import json
import torch
from PIL import Image
from datasets import load_dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from jiwer import cer, wer


## Load Dataset

In [ ]:
print("Loading dataset...")
baseline_dataset = load_dataset("Teklia/IAM-line", split="test[:1000]")

# Preview
sample = baseline_dataset[0]
sample["image"].save("temp_trocr.jpg")
print("GT contoh pertama:", sample["text"])
sample["image"]


## Normalisasi Teks
> Sama persis dengan DeepSeek OCR2 baseline

In [ ]:
def normalize_text(text):
    text = text.strip()
    text = text.replace("
", " ")       # newline → space
    text = re.sub(r" +", " ", text)      # multi-space → single space
    text = text.lower()                  # lowercase (case-insensitive)
    return text


## Load Model TrOCR

In [ ]:
MODEL_NAME = "microsoft/trocr-large-handwritten"

print(f"Loading {MODEL_NAME}...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print(f"Model loaded on: {device}")


## Evaluasi TrOCR Baseline
> Loop identik dengan DeepSeek OCR2: per-sample CER/WER, akumulasi, lalu rata-rata

In [ ]:
baseline_total_cer = 0
baseline_total_wer = 0

for i, sample in enumerate(baseline_dataset):
    image = sample["image"]
    if image.mode != "RGB":
        image = image.convert("RGB")

    gt = normalize_text(sample["text"])

    # ---- Inferensi TrOCR ----
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    pred_raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    pred = normalize_text(pred_raw)

    # ---- Hitung Metrik (identik DeepSeek) ----
    baseline_total_cer += cer(gt, pred)
    baseline_total_wer += wer(gt, pred)

    if (i + 1) % 50 == 0 or i == 0:
        print(f"[{i+1}/{len(baseline_dataset)}]")
        print(f"  GT  : {gt}")
        print(f"  Pred: {pred}")
        print(f"  CER : {cer(gt, pred):.4f} | WER : {wer(gt, pred):.4f}
")

print(f"Done {i+1}/{len(baseline_dataset)}

")


## Hasil & Simpan

In [ ]:
# Hitung rata-rata  (sama persis dengan DeepSeek OCR2)
baseline_avg_cer = baseline_total_cer / len(baseline_dataset)
baseline_avg_wer = baseline_total_wer / len(baseline_dataset)

print("
=== TROCR BASELINE RESULT ===")
print("Baseline CER:", baseline_avg_cer)
print("Baseline WER:", baseline_avg_wer)

# ---- Simpan TXT ----
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)

with open(f"{output_dir}/trocr_baseline_results.txt", "w") as f:
    f.write("=== TROCR BASELINE RESULT ===
")
    f.write(f"Model       : {MODEL_NAME}
")
    f.write(f"Baseline CER: {baseline_avg_cer}
")
    f.write(f"Baseline WER: {baseline_avg_wer}
")
print(f"Hasil disimpan ke {output_dir}/trocr_baseline_results.txt")

# ---- Simpan JSON ----
baseline_results = {
    "model": MODEL_NAME,
    "test_split": "test[:1000]",
    "num_samples": len(baseline_dataset),
    "avg_cer": baseline_avg_cer,
    "avg_wer": baseline_avg_wer,
}
with open(f"{output_dir}/trocr_baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)
print(f"Hasil disimpan ke {output_dir}/trocr_baseline_results.json")
